# Minimum Spanning Tree Regimes

Uses a minimum spanning tree to organize portfolio exposure by regime.

This notebook downloads daily historical market prices from Yahoo Finance. Results depend on the configured dates and symbols.


In [ ]:
from _market_data import download_price_records
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import warnings

warnings.filterwarnings("ignore")
from scipy.spatial.distance import squareform, pdist
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster
from scipy.stats import norm, t
import scipy.optimize as sco
import networkx as nx
from networkx.algorithms import tree
import matplotlib.pyplot as plt
import seaborn as sns

print("=" * 80)
print("REGIME-ADAPTIVE PORTFOLIO ALLOCATION FRAMEWORK")
print("MST Entropy + Momentum-OPS + HRP + Risk Parity")
print("MODIFIED: 14-day rebalancing + Fixed regime detection")
print("=" * 80)
print()
CONFIG = {
    "ETF_UNIVERSE": [
        "SPY",
        "QQQ",
        "IWM",
        "DIA",
        "EFA",
        "EEM",
        "VGK",
        "TLT",
        "SHY",
        "LQD",
        "HYG",
        "GLD",
        "SLV",
        "DBC",
        "USO",
        "SH",
        "VNQ",
        "VXX",
    ],
    "BENCHMARK": "URTH",
    "START_DATE": "2014-01-01",
    "END_DATE": "2018-12-31",
    "REGIME_WINDOW": 252,
    "REGIME_QUANTILES": {"BEARISH_THRESHOLD": 0.33, "BULLISH_THRESHOLD": 0.67},
    "REGIME_MIN_DURATION": 5,
    "MOMENTUM_LOOKBACK": 252,
    "MOMENTUM_SKIP": 21,
    "MOMENTUM_TOP_N": 5,
    "RP_TARGET_VOL": 0.1,
    "RP_VOL_WINDOW": 60,
    "RP_MAX_SH_ALLOCATION": 0.2,
    "REBALANCE_FREQ": "14D",
    "INITIAL_CAPITAL": 1000000,
    "COMPOUNDING_MODE": True,
    "LEVERAGE": 1.0,
    "COMMISSION_BPS": 10,
    "TRAIN_YEARS": 2,
    "TEST_MONTHS": 6,
    "MIN_DATA_POINTS": 500,
    "MAX_POSITION_SIZE": 0.25,
    "MIN_POSITION_SIZE": 0.02,
}
print("Configuration loaded:")
print(f"  ETF Universe: {len(CONFIG['ETF_UNIVERSE'])} assets")
print(f"  Date Range: {CONFIG['START_DATE']} to {CONFIG['END_DATE']}")
print(f"  Regime Window: {CONFIG['REGIME_WINDOW']} days")
print(f"  Rebalance Frequency: {CONFIG['REBALANCE_FREQ']} (14 days) - MODIFIED")
print(f"  Initial Capital: ${CONFIG['INITIAL_CAPITAL']:,.0f}")
print(f"  Compounding Mode: {CONFIG['COMPOUNDING_MODE']}")
print(f"  Transaction Cost: {CONFIG['COMMISSION_BPS']} bps")
print()


def fetch_yahoo_data(ticker, start_date, end_date):
    """
    Fetch historical daily data from Yahoo Finance

    Parameters:
    -----------
    ticker : str
        ETF ticker symbol
    start_date : str
        Start date in 'YYYY-MM-DD' format
    end_date : str
        End date in 'YYYY-MM-DD' format
        Yahoo Finance key

    Returns:
    --------
    pd.DataFrame
        DataFrame with Date index and adjusted close prices
    """
    try:
        response_prices = {"historical": download_price_records(ticker, start_date, end_date)}
        data = response_prices
        if "historical" not in data:
            print(f"  WARNING: No data for {ticker}")
            return pd.DataFrame()
        df = pd.DataFrame(data["historical"])
        df["date"] = pd.to_datetime(df["date"])
        df = df.sort_values("date")
        df = df.set_index("date")
        df = df[["adjClose"]].rename(columns={"adjClose": ticker})
        return df
    except Exception as e:
        print(f"  ERROR fetching {ticker}: {str(e)}")
        return pd.DataFrame()


def download_all_data(tickers, start_date, end_date):
    """
    Download data for all tickers and merge into single DataFrame

    CRITICAL: Only keeps assets with complete data (no forward/backward fill)
    to avoid artificial data that could introduce look-ahead bias
    """
    print("Downloading price data from Yahoo Finance...")
    all_data = []
    for ticker in tickers:
        print(f"  Fetching {ticker}...", end=" ")
        df = fetch_yahoo_data(ticker, start_date, end_date)
        if not df.empty:
            all_data.append(df)
            print(f"✓ {len(df)} days")
        else:
            print("✗ FAILED")
    if len(all_data) == 0:
        raise ValueError("No data downloaded. Check API key and tickers.")
    prices = pd.concat(all_data, axis=1, join="outer")
    missing_pct = prices.isnull().sum() / len(prices)
    print()
    print("Data Quality Report:")
    print("-" * 60)
    print(f"Date range: {prices.index.min()} to {prices.index.max()}")
    print(f"Total trading days: {len(prices)}")
    print(f"Assets downloaded: {prices.shape[1]}")
    print()
    if missing_pct.max() > 0:
        print("WARNING: Missing data detected:")
        for ticker in missing_pct[missing_pct > 0].index:
            print(f"  {ticker}: {missing_pct[ticker] * 100:.1f}% missing")
        print()
        threshold = 0.05
        to_drop = missing_pct[missing_pct > threshold].index.tolist()
        if to_drop:
            print(f"  Dropping assets with >5% missing: {to_drop}")
            prices = prices.drop(columns=to_drop)
    print(f"Final universe: {prices.shape[1]} assets")
    print("=" * 60)
    print()
    return prices


def compute_correlation_distance(returns):
    """
    Convert correlation matrix to distance matrix

    Distance formula: d_ij = sqrt(2 * (1 - corr_ij))
    This ensures d_ij ∈ [0, 2] with perfect correlation → 0 distance

    Parameters:
    -----------
    returns : pd.DataFrame
        Asset returns

    Returns:
    --------
    np.ndarray
        Distance matrix (symmetric)
    """
    returns_clean = returns.dropna(axis=1, how="all")
    corr_matrix = returns_clean.corr()
    distance_matrix = np.sqrt(2 * (1 - corr_matrix))
    distance_matrix = np.clip(distance_matrix, 0, 2)
    return (distance_matrix.values, returns_clean.columns.tolist())


def build_mst(distance_matrix, asset_names):
    """
    Build Minimum Spanning Tree using Kruskal's algorithm

    Parameters:
    -----------
    distance_matrix : np.ndarray
        Distance matrix (symmetric)
    asset_names : list
        List of asset names

    Returns:
    --------
    nx.Graph
        MST as NetworkX graph
    """
    G = nx.Graph()
    n_assets = len(asset_names)
    for i in range(n_assets):
        for j in range(i + 1, n_assets):
            G.add_edge(asset_names[i], asset_names[j], weight=distance_matrix[i, j])
    mst = tree.minimum_spanning_tree(G, algorithm="kruskal")
    return mst


def compute_mst_entropy(mst):
    """
    Compute Shannon entropy of MST based on node degree distribution

    High entropy → nodes have similar degrees → decentralized network → BULLISH
    Low entropy → hub-and-spoke structure → centralized network → BEARISH

    Entropy formula: H = -sum(p_i * log(p_i))
    where p_i = degree_i / sum(all degrees)

    Parameters:
    -----------
    mst : nx.Graph
        Minimum spanning tree

    Returns:
    --------
    float
        Shannon entropy
    dict
        Additional MST metrics
    """
    degrees = dict(mst.degree())
    degree_values = np.array(list(degrees.values()))
    total_degree = degree_values.sum()
    if total_degree == 0:
        return (0.0, {})
    probabilities = degree_values / total_degree
    probabilities = probabilities[probabilities > 0]
    entropy = -np.sum(probabilities * np.log(probabilities))
    metrics = {
        "entropy": entropy,
        "avg_degree": degree_values.mean(),
        "max_degree": degree_values.max(),
        "tree_length": sum([data["weight"] for _, _, data in mst.edges(data=True)]),
        "n_nodes": len(mst.nodes),
        "normalized_entropy": entropy / np.log(len(mst.nodes)) if len(mst.nodes) > 1 else 0,
    }
    return (entropy, metrics)


def detect_regime_mst(returns, window, thresholds_dict=None):
    """
    Detect market regime using rolling MST entropy

    CRITICAL MODIFICATION: Now uses NORMALIZED entropy instead of raw entropy
    This fixes the bearish bias caused by entropy scaling with number of nodes

    CRITICAL: Uses T-1 data to classify regime at T (no look-ahead)

    Parameters:
    -----------
    returns : pd.DataFrame
        Asset returns
    window : int
        Rolling window size for MST computation
    thresholds_dict : dict, optional
        Dictionary with 'bearish' and 'bullish' entropy thresholds
        If None, uses expanding window to compute quantiles dynamically

    Returns:
    --------
    pd.Series
        Regime labels: 'BULLISH', 'NEUTRAL', 'BEARISH'
    pd.DataFrame
        MST metrics over time
    """
    entropy_series = []
    dates = []
    for i in range(window, len(returns)):
        window_returns = returns.iloc[i - window : i]
        try:
            dist_matrix, assets = compute_correlation_distance(window_returns)
            mst = build_mst(dist_matrix, assets)
            entropy, metrics = compute_mst_entropy(mst)
            entropy_series.append(metrics)
            dates.append(returns.index[i])
        except Exception as e:
            entropy_series.append(
                {
                    "entropy": np.nan,
                    "normalized_entropy": np.nan,
                    "tree_length": np.nan,
                    "avg_degree": np.nan,
                    "max_degree": np.nan,
                    "n_nodes": 0,
                }
            )
            dates.append(returns.index[i])
    mst_metrics = pd.DataFrame(entropy_series, index=dates)
    regimes = pd.Series(index=dates, dtype=str)
    for i in range(len(mst_metrics)):
        entropy_history = mst_metrics["normalized_entropy"].iloc[: i + 1].dropna()
        if len(entropy_history) < 20:
            regimes.iloc[i] = "NEUTRAL"
            continue
        bearish_threshold = entropy_history.quantile(0.33)
        bullish_threshold = entropy_history.quantile(0.67)
        current_entropy = mst_metrics["normalized_entropy"].iloc[i]
        if pd.isna(current_entropy):
            regimes.iloc[i] = "NEUTRAL"
        elif current_entropy <= bearish_threshold:
            regimes.iloc[i] = "BEARISH"
        elif current_entropy >= bullish_threshold:
            regimes.iloc[i] = "BULLISH"
        else:
            regimes.iloc[i] = "NEUTRAL"
    return (regimes, mst_metrics)


def smooth_regimes(regimes, min_duration):
    """
    Smooth regime transitions to prevent whipsaw

    CRITICAL: Prevents frequent regime switching that would generate
    excessive transaction costs

    Parameters:
    -----------
    regimes : pd.Series
        Raw regime labels
    min_duration : int
        Minimum number of days to hold a regime

    Returns:
    --------
    pd.Series
        Smoothed regime labels
    """
    smoothed = regimes.copy()
    current_regime = regimes.iloc[0]
    regime_start = 0
    for i in range(1, len(regimes)):
        if regimes.iloc[i] != current_regime:
            duration = i - regime_start
            if duration < min_duration:
                smoothed.iloc[i] = current_regime
            else:
                current_regime = regimes.iloc[i]
                regime_start = i
        else:
            smoothed.iloc[i] = current_regime
    return smoothed


def diagnose_regime_distribution(mst_metrics, regimes):
    """
    Diagnostic function to understand regime classification

    This helps identify issues with regime detection and validates
    that the normalized entropy fix is working properly

    Parameters:
    -----------
    mst_metrics : pd.DataFrame
        MST metrics over time
    regimes : pd.Series
        Regime labels
    """
    print("\n" + "=" * 80)
    print("REGIME DISTRIBUTION DIAGNOSTICS")
    print("=" * 80)
    print("\nRAW ENTROPY STATISTICS:")
    print(mst_metrics["entropy"].describe())
    print("\nNORMALIZED ENTROPY STATISTICS (USED FOR REGIME DETECTION):")
    print(mst_metrics["normalized_entropy"].describe())
    print("\nTREE LENGTH STATISTICS:")
    print(mst_metrics["tree_length"].describe())
    norm_entropy = mst_metrics["normalized_entropy"].dropna()
    print(f"\nACTUAL THRESHOLDS (Normalized Entropy):")
    print(f"  Bearish threshold (33rd percentile): {norm_entropy.quantile(0.33):.4f}")
    print(f"  Neutral range: {norm_entropy.quantile(0.33):.4f} - {norm_entropy.quantile(0.67):.4f}")
    print(f"  Bullish threshold (67th percentile): {norm_entropy.quantile(0.67):.4f}")
    regime_map = {"BEARISH": -1, "NEUTRAL": 0, "BULLISH": 1}
    regime_numeric = regimes.map(regime_map)
    print(f"\nCORRELATIONS WITH REGIME:")
    print(
        f"  Normalized Entropy vs Regime: {mst_metrics['normalized_entropy'].corr(regime_numeric):.4f}"
    )
    print(f"  Tree Length vs Regime: {mst_metrics['tree_length'].corr(regime_numeric):.4f}")
    print(f"  Avg Degree vs Regime: {mst_metrics['avg_degree'].corr(regime_numeric):.4f}")
    merged = pd.DataFrame(
        {
            "regime": regimes,
            "norm_entropy": mst_metrics["normalized_entropy"],
            "raw_entropy": mst_metrics["entropy"],
            "tree_length": mst_metrics["tree_length"],
            "avg_degree": mst_metrics["avg_degree"],
        }
    )
    print("\nAVERAGE METRICS BY REGIME:")
    print(
        merged.groupby("regime")
        .agg(
            {
                "norm_entropy": ["mean", "std"],
                "tree_length": ["mean", "std"],
                "avg_degree": ["mean", "std"],
            }
        )
        .round(4)
    )
    print("\nREGIME COUNTS:")
    print(regimes.value_counts())
    print(f"\nREGIME PROPORTIONS:")
    print((regimes.value_counts() / len(regimes) * 100).round(2))
    print("=" * 80 + "\n")


def compute_momentum_scores(prices, lookback, skip, date):
    """
    Compute momentum scores for all assets at a given date

    Uses standard 12-1 month momentum:
    - lookback = 252 days (12 months)
    - skip = 21 days (1 month) to avoid short-term reversal

    CRITICAL: Uses only data up to T-1 (no look-ahead)

    Parameters:
    -----------
    prices : pd.DataFrame
        Asset prices
    lookback : int
        Momentum lookback period
    skip : int
        Days to skip at end
    date : pd.Timestamp
        Current date

    Returns:
    --------
    pd.Series
        Momentum scores (total return from T-lookback to T-skip)
    """
    date_idx = prices.index.get_loc(date)
    if date_idx < lookback:
        return pd.Series(dtype=float)
    end_idx = date_idx - skip
    start_idx = date_idx - lookback
    if start_idx < 0 or end_idx < skip:
        return pd.Series(dtype=float)
    prices_start = prices.iloc[start_idx]
    prices_end = prices.iloc[end_idx]
    momentum = prices_end / prices_start - 1
    return momentum.dropna()


def momentum_ops_allocation(prices, date, config, available_assets):
    """
    Online Portfolio Selection - Momentum Strategy

    Select top N assets by momentum and equal-weight them

    Parameters:
    -----------
    prices : pd.DataFrame
        Asset prices
    date : pd.Timestamp
        Current date
    config : dict
        Configuration dictionary
    available_assets : list
        List of assets available for allocation (excludes SH typically)

    Returns:
    --------
    pd.Series
        Portfolio weights
    """
    momentum = compute_momentum_scores(
        prices[available_assets], config["MOMENTUM_LOOKBACK"], config["MOMENTUM_SKIP"], date
    )
    if len(momentum) == 0:
        weights = pd.Series(1.0 / len(available_assets), index=available_assets)
        return weights
    top_assets = momentum.nlargest(config["MOMENTUM_TOP_N"]).index.tolist()
    weights = pd.Series(0.0, index=available_assets)
    weights[top_assets] = 1.0 / len(top_assets)
    return weights


def hrp_allocation(returns, date, window=252):
    """
    Hierarchical Risk Parity allocation

    Algorithm:
    1. Compute correlation matrix
    2. Hierarchical clustering using Ward linkage
    3. Quasi-diagonalization to group similar assets
    4. Recursive bisection with inverse-variance weighting

    CRITICAL: Uses only data up to T-1 (no look-ahead)

    Parameters:
    -----------
    returns : pd.DataFrame
        Asset returns
    date : pd.Timestamp
        Current date
    window : int
        Lookback window for covariance estimation

    Returns:
    --------
    pd.Series
        Portfolio weights
    """
    date_idx = returns.index.get_loc(date)
    if date_idx < window:
        return pd.Series(1.0 / len(returns.columns), index=returns.columns)
    window_returns = returns.iloc[date_idx - window : date_idx]
    window_returns = window_returns.dropna(axis=1, how="any")
    if len(window_returns.columns) < 2:
        return pd.Series(1.0 / len(returns.columns), index=returns.columns)
    corr = window_returns.corr()
    dist = np.sqrt((1 - corr) / 2)
    dist_condensed = squareform(dist.values, checks=False)
    link = linkage(dist_condensed, method="ward")
    sort_idx = _quasi_diagonalize(link, dist.shape[0])
    weights = _recursive_bisection(window_returns.iloc[:, sort_idx].cov())
    asset_order = dist.columns[sort_idx].tolist()
    weights_series = pd.Series(weights, index=asset_order)
    weights_full = pd.Series(0.0, index=returns.columns)
    weights_full[weights_series.index] = weights_series
    return weights_full


def _quasi_diagonalize(link, n_assets):
    """
    Quasi-diagonalize linkage matrix to group similar assets

    Returns indices that reorder the distance matrix
    """
    from scipy.cluster.hierarchy import dendrogram

    dend = dendrogram(link, no_plot=True)
    return dend["leaves"]


def _recursive_bisection(cov):
    """
    Recursive bisection with inverse-variance weighting

    Parameters:
    -----------
    cov : pd.DataFrame
        Covariance matrix

    Returns:
    --------
    np.ndarray
        Weights
    """
    n_assets = cov.shape[0]
    weights = pd.Series(1.0, index=cov.index)
    clusters = [cov.index.tolist()]
    while len(clusters) > 0:
        new_clusters = []
        for cluster in clusters:
            if len(cluster) == 1:
                continue
            split_idx = len(cluster) // 2
            cluster1 = cluster[:split_idx]
            cluster2 = cluster[split_idx:]
            cov1 = cov.loc[cluster1, cluster1]
            cov2 = cov.loc[cluster2, cluster2]
            w1 = np.ones(len(cluster1)) / len(cluster1)
            w2 = np.ones(len(cluster2)) / len(cluster2)
            var1 = np.dot(w1, np.dot(cov1.values, w1))
            var2 = np.dot(w2, np.dot(cov2.values, w2))
            alpha = 1 - var1 / (var1 + var2)
            weights[cluster1] *= alpha
            weights[cluster2] *= 1 - alpha
            if len(cluster1) > 1:
                new_clusters.append(cluster1)
            if len(cluster2) > 1:
                new_clusters.append(cluster2)
        clusters = new_clusters
    return weights.values


def risk_parity_allocation(returns, date, window=60, target_vol=0.1, max_sh=0.2):
    """
    Risk Parity allocation with optional SH hedge

    Allocates based on inverse volatility, then scales to target volatility
    Allows up to max_sh allocation to SH (inverse ETF)

    CRITICAL: Uses only data up to T-1 (no look-ahead)

    Parameters:
    -----------
    returns : pd.DataFrame
        Asset returns
    date : pd.Timestamp
        Current date
    window : int
        Volatility lookback window
    target_vol : float
        Target portfolio volatility (annualized)
    max_sh : float
        Maximum allocation to SH

    Returns:
    --------
    pd.Series
        Portfolio weights
    """
    date_idx = returns.index.get_loc(date)
    if date_idx < window:
        return pd.Series(1.0 / len(returns.columns), index=returns.columns)
    window_returns = returns.iloc[date_idx - window : date_idx]
    window_returns = window_returns.dropna(axis=1, how="any")
    if len(window_returns.columns) < 2:
        return pd.Series(1.0 / len(returns.columns), index=returns.columns)
    vols = window_returns.std() * np.sqrt(252)
    inv_vols = 1.0 / vols
    weights = inv_vols / inv_vols.sum()
    if "SH" in weights.index:
        if weights["SH"] > max_sh:
            excess = weights["SH"] - max_sh
            weights["SH"] = max_sh
            other_assets = weights.index[weights.index != "SH"]
            if len(other_assets) > 0:
                weights[other_assets] += excess * (
                    weights[other_assets] / weights[other_assets].sum()
                )
    weights_full = pd.Series(0.0, index=returns.columns)
    weights_full[weights.index] = weights
    return weights_full


def regime_adaptive_allocation(regime, prices, returns, date, config):
    """
    Allocate portfolio based on current regime

    BULLISH: 80% Momentum-OPS + 20% HRP
    NEUTRAL: 60% HRP + 40% Momentum-OPS
    BEARISH: 100% Risk Parity (with up to 20% SH)

    Parameters:
    -----------
    regime : str
        Current regime ('BULLISH', 'NEUTRAL', 'BEARISH')
    prices : pd.DataFrame
        Asset prices
    returns : pd.DataFrame
        Asset returns
    date : pd.Timestamp
        Current date
    config : dict
        Configuration dictionary

    Returns:
    --------
    pd.Series
        Portfolio weights
    str
        Allocation method used
    """
    long_assets = [a for a in returns.columns if a not in ["SH", "VXX"]]
    if regime == "BULLISH":
        momentum_weights = momentum_ops_allocation(prices, date, config, long_assets)
        hrp_weights = hrp_allocation(returns[long_assets], date)
        final_weights = 0.8 * momentum_weights + 0.2 * hrp_weights
        allocation_method = "BULLISH: 80% Momentum-OPS + 20% HRP"
    elif regime == "NEUTRAL":
        momentum_weights = momentum_ops_allocation(prices, date, config, long_assets)
        hrp_weights = hrp_allocation(returns[long_assets], date)
        final_weights = 0.4 * momentum_weights + 0.6 * hrp_weights
        allocation_method = "NEUTRAL: 60% HRP + 40% Momentum-OPS"
    elif regime == "BEARISH":
        final_weights = risk_parity_allocation(
            returns,
            date,
            window=config["RP_VOL_WINDOW"],
            target_vol=config["RP_TARGET_VOL"],
            max_sh=config["RP_MAX_SH_ALLOCATION"],
        )
        allocation_method = "BEARISH: Risk Parity with SH hedge"
    else:
        final_weights = pd.Series(1.0 / len(returns.columns), index=returns.columns)
        allocation_method = "UNKNOWN: Equal Weight"
    final_weights = final_weights.clip(lower=0, upper=config["MAX_POSITION_SIZE"])
    final_weights[final_weights < config["MIN_POSITION_SIZE"]] = 0
    if final_weights.sum() > 0:
        final_weights = final_weights / final_weights.sum()
    else:
        final_weights = pd.Series(1.0 / len(returns.columns), index=returns.columns)
    return (final_weights, allocation_method)


def compute_transaction_costs(
    weights_old, weights_new, prices_current, portfolio_value, commission_bps
):
    """
    Compute transaction costs from rebalancing

    Cost = sum(|weight_new - weight_old| * portfolio_value * commission_rate)

    Parameters:
    -----------
    weights_old : pd.Series
        Old weights
    weights_new : pd.Series
        New weights
    prices_current : pd.Series
        Current prices
    portfolio_value : float
        Current portfolio value
    commission_bps : float
        Commission in basis points

    Returns:
    --------
    float
        Total transaction cost
    """
    all_assets = weights_old.index.union(weights_new.index)
    w_old = weights_old.reindex(all_assets, fill_value=0)
    w_new = weights_new.reindex(all_assets, fill_value=0)
    turnover = np.abs(w_new - w_old).sum()
    cost = turnover * portfolio_value * (commission_bps / 10000)
    return cost


def backtest_strategy(prices, returns, regimes, config):
    """
    Backtest the regime-adaptive strategy

    CRITICAL TEMPORAL LOGIC:
    ------------------------
    Day T-1 (Close):
        - Portfolio valued at Close[T-1]
        - Compute regime using data up to T-1
        - Generate allocation signals
        - Store target weights

    Day T (Open):
        - Execute rebalance at Open[T] ≈ Close[T-1]
        - Pay transaction costs
        - Update positions

    Day T (Close):
        - Portfolio valued at Close[T]
        - Returns = Close[T]/Close[T-1] - 1
        - This becomes Close[T-1] for next period

    FIXED: Track portfolio values with proper dates

    Parameters:
    -----------
    prices : pd.DataFrame
        Asset prices
    returns : pd.DataFrame
        Asset returns
    regimes : pd.Series
        Regime labels
    config : dict
        Configuration dictionary

    Returns:
    --------
    dict
        Backtest results
    """
    print()
    print("=" * 80)
    print("RUNNING BACKTEST")
    print("=" * 80)
    rebalance_dates = returns.resample(config["REBALANCE_FREQ"]).last().index
    rebalance_dates = rebalance_dates[rebalance_dates.isin(regimes.index)]
    portfolio_values = []
    portfolio_weights = []
    portfolio_dates = []
    allocation_methods = []
    regime_history = []
    transaction_costs = []
    if config["COMPOUNDING_MODE"]:
        portfolio_value = config["INITIAL_CAPITAL"]
    else:
        portfolio_value = config["INITIAL_CAPITAL"]
        notional_capital = config["INITIAL_CAPITAL"]
    current_weights = pd.Series(0.0, index=returns.columns)
    print(f"\nInitial capital: ${portfolio_value:,.0f}")
    print(f"Compounding mode: {config['COMPOUNDING_MODE']}")
    print(f"Rebalance dates: {len(rebalance_dates)}")
    print()
    for i, date in enumerate(rebalance_dates):
        if date not in regimes.index:
            continue
        current_regime = regimes.loc[date]
        new_weights, allocation_method = regime_adaptive_allocation(
            current_regime, prices.loc[:date], returns.loc[:date], date, config
        )
        if i > 0:
            cost = compute_transaction_costs(
                current_weights,
                new_weights,
                prices.loc[date],
                portfolio_value if config["COMPOUNDING_MODE"] else notional_capital,
                config["COMMISSION_BPS"],
            )
            transaction_costs.append(cost)
            if config["COMPOUNDING_MODE"]:
                portfolio_value -= cost
            else:
                pass
        current_weights = new_weights
        portfolio_weights.append(current_weights.to_dict())
        portfolio_dates.append(date)
        allocation_methods.append(allocation_method)
        regime_history.append(current_regime)
        if i < len(rebalance_dates) - 1:
            next_date = rebalance_dates[i + 1]
        else:
            next_date = returns.index[-1]
        period_returns = returns.loc[date:next_date]
        for ret_date in period_returns.index[1:]:
            daily_return = (current_weights * period_returns.loc[ret_date]).sum()
            if config["COMPOUNDING_MODE"]:
                portfolio_value *= 1 + daily_return
            else:
                portfolio_value = notional_capital * (1 + daily_return)
            portfolio_values.append((ret_date, portfolio_value))
        if (i + 1) % 20 == 0:
            print(
                f"  Processed {i + 1}/{len(rebalance_dates)} rebalances - "
                + f"Value: ${portfolio_value:,.0f} - Regime: {current_regime}"
            )
    print()
    print(f"Backtest complete!")
    print(f"Final portfolio value: ${portfolio_value:,.0f}")
    print(f"Total return: {(portfolio_value / config['INITIAL_CAPITAL'] - 1) * 100:.2f}%")
    print(f"Total transaction costs: ${sum(transaction_costs):,.0f}")
    print()
    weights_df = pd.DataFrame(portfolio_weights, index=portfolio_dates)
    pv_dates = [pv[0] for pv in portfolio_values]
    pv_values = [pv[1] for pv in portfolio_values]
    portfolio_values_series = pd.Series(pv_values, index=pv_dates)
    results = {
        "portfolio_value": portfolio_value,
        "portfolio_values_series": portfolio_values_series,
        "weights": weights_df,
        "regimes": pd.Series(regime_history, index=portfolio_dates),
        "allocation_methods": pd.Series(allocation_methods, index=portfolio_dates),
        "transaction_costs": pd.Series(transaction_costs, index=portfolio_dates[1:]),
        "config": config,
    }
    return results


def compute_performance_metrics(returns, risk_free_rate=0.02):
    """
    Compute comprehensive performance metrics

    Parameters:
    -----------
    returns : pd.Series
        Portfolio returns
    risk_free_rate : float
        Annual risk-free rate

    Returns:
    --------
    dict
        Performance metrics
    """
    total_return = (1 + returns).prod() - 1
    annualized_return = (1 + total_return) ** (252 / len(returns)) - 1
    annualized_vol = returns.std() * np.sqrt(252)
    excess_returns = returns - risk_free_rate / 252
    sharpe = excess_returns.mean() / returns.std() * np.sqrt(252)
    cumulative = (1 + returns).cumprod()
    running_max = cumulative.expanding().max()
    drawdown = (cumulative - running_max) / running_max
    max_drawdown = drawdown.min()
    calmar = annualized_return / abs(max_drawdown) if max_drawdown != 0 else 0
    downside_returns = returns[returns < 0]
    downside_std = downside_returns.std() * np.sqrt(252)
    sortino = (annualized_return - risk_free_rate) / downside_std if downside_std > 0 else 0
    win_rate = (returns > 0).sum() / len(returns)
    n = len(returns)
    skew = returns.skew()
    kurt = returns.kurtosis()
    sr_std = np.sqrt((1 + 0.5 * sharpe**2 - skew * sharpe + (kurt - 3) / 4 * sharpe**2) / n)
    psr = norm.cdf(sharpe / sr_std) if sr_std > 0 else 0
    dsr = sharpe / np.sqrt(1 + 0.5 * sharpe**2)
    metrics = {
        "Total Return": total_return,
        "Annualized Return": annualized_return,
        "Annualized Volatility": annualized_vol,
        "Sharpe Ratio": sharpe,
        "Sortino Ratio": sortino,
        "Calmar Ratio": calmar,
        "Max Drawdown": max_drawdown,
        "Win Rate": win_rate,
        "Skewness": skew,
        "Kurtosis": kurt,
        "Probabilistic Sharpe Ratio": psr,
        "Deflated Sharpe Ratio": dsr,
    }
    return metrics


def compare_with_benchmark(strategy_returns, benchmark_returns):
    """
    Compare strategy with benchmark

    Parameters:
    -----------
    strategy_returns : pd.Series
        Strategy returns
    benchmark_returns : pd.Series
        Benchmark returns

    Returns:
    --------
    dict
        Comparison metrics
    """
    aligned = pd.DataFrame({"Strategy": strategy_returns, "Benchmark": benchmark_returns}).dropna()
    strategy_metrics = compute_performance_metrics(aligned["Strategy"])
    benchmark_metrics = compute_performance_metrics(aligned["Benchmark"])
    cov_matrix = aligned.cov()
    beta = cov_matrix.loc["Strategy", "Benchmark"] / cov_matrix.loc["Benchmark", "Benchmark"]
    alpha = strategy_metrics["Annualized Return"] - benchmark_metrics["Annualized Return"] * beta
    tracking_error = (aligned["Strategy"] - aligned["Benchmark"]).std() * np.sqrt(252)
    information_ratio = alpha / tracking_error if tracking_error > 0 else 0
    comparison = {
        "Strategy": strategy_metrics,
        "Benchmark": benchmark_metrics,
        "Alpha": alpha,
        "Beta": beta,
        "Information Ratio": information_ratio,
        "Tracking Error": tracking_error,
    }
    return comparison


def plot_results(results, benchmark_returns, mst_metrics):
    """
    Create comprehensive visualization of backtest results

    Parameters:
    -----------
    results : dict
        Backtest results
    benchmark_returns : pd.Series
        Benchmark returns
    mst_metrics : pd.DataFrame
        MST metrics over time
    """
    plt.style.use("seaborn-v0_8-darkgrid")
    sns.set_palette("husl")
    fig = plt.figure(figsize=(20, 16))
    gs = fig.add_gridspec(5, 2, hspace=0.3, wspace=0.3)
    ax1 = fig.add_subplot(gs[0, :])
    portfolio_values = results["portfolio_values_series"]
    portfolio_cumulative = portfolio_values / results["config"]["INITIAL_CAPITAL"]
    benchmark_aligned = benchmark_returns.reindex(portfolio_values.index).fillna(0)
    benchmark_cumulative = (1 + benchmark_aligned).cumprod()
    ax1.plot(
        portfolio_cumulative.index,
        portfolio_cumulative.values,
        label="Strategy",
        linewidth=2,
        color="#2E86AB",
    )
    ax1.plot(
        benchmark_cumulative.index,
        benchmark_cumulative.values,
        label="Benchmark (URTH)",
        linewidth=2,
        color="#A23B72",
        alpha=0.7,
    )
    ax1.set_title("Cumulative Portfolio Performance", fontsize=16, fontweight="bold")
    ax1.set_xlabel("Date", fontsize=12)
    ax1.set_ylabel("Cumulative Return", fontsize=12)
    ax1.legend(loc="upper left", fontsize=11)
    ax1.grid(True, alpha=0.3)
    ax2 = fig.add_subplot(gs[1, :])
    running_max = portfolio_cumulative.expanding().max()
    drawdown = (portfolio_cumulative - running_max) / running_max
    ax2.fill_between(
        drawdown.index, 0, drawdown.values * 100, alpha=0.3, color="red", label="Drawdown"
    )
    ax2.plot(drawdown.index, drawdown.values * 100, color="darkred", linewidth=1.5)
    ax2.set_title("Strategy Drawdown", fontsize=16, fontweight="bold")
    ax2.set_xlabel("Date", fontsize=12)
    ax2.set_ylabel("Drawdown (%)", fontsize=12)
    ax2.legend(loc="lower left", fontsize=11)
    ax2.grid(True, alpha=0.3)
    ax3 = fig.add_subplot(gs[2, :])
    regimes = results["regimes"]
    regime_colors = {"BULLISH": "green", "NEUTRAL": "gray", "BEARISH": "red"}
    regime_numeric = regimes.map({"BULLISH": 1, "NEUTRAL": 0, "BEARISH": -1})
    for i in range(len(regime_numeric) - 1):
        ax3.axvspan(
            regime_numeric.index[i],
            regime_numeric.index[i + 1],
            alpha=0.3,
            color=regime_colors[regimes.iloc[i]],
        )
    ax3.plot(
        regime_numeric.index,
        regime_numeric.values,
        color="black",
        linewidth=2,
        marker="o",
        markersize=3,
    )
    ax3.set_title("Market Regime Timeline", fontsize=16, fontweight="bold")
    ax3.set_xlabel("Date", fontsize=12)
    ax3.set_ylabel("Regime", fontsize=12)
    ax3.set_yticks([-1, 0, 1])
    ax3.set_yticklabels(["BEARISH", "NEUTRAL", "BULLISH"])
    ax3.grid(True, alpha=0.3)
    from matplotlib.patches import Patch

    legend_elements = [
        Patch(facecolor="green", alpha=0.3, label="Bullish"),
        Patch(facecolor="gray", alpha=0.3, label="Neutral"),
        Patch(facecolor="red", alpha=0.3, label="Bearish"),
    ]
    ax3.legend(handles=legend_elements, loc="upper right", fontsize=11)
    ax4 = fig.add_subplot(gs[3, 0])
    ax4.plot(mst_metrics.index, mst_metrics["normalized_entropy"], color="purple", linewidth=1.5)
    ax4.set_title("MST Normalized Entropy", fontsize=14, fontweight="bold")
    ax4.set_xlabel("Date", fontsize=11)
    ax4.set_ylabel("Normalized Entropy", fontsize=11)
    ax4.grid(True, alpha=0.3)
    ax5 = fig.add_subplot(gs[3, 1])
    ax5.plot(mst_metrics.index, mst_metrics["tree_length"], color="orange", linewidth=1.5)
    ax5.set_title("MST Tree Length Over Time", fontsize=14, fontweight="bold")
    ax5.set_xlabel("Date", fontsize=11)
    ax5.set_ylabel("Tree Length", fontsize=11)
    ax5.grid(True, alpha=0.3)
    ax6 = fig.add_subplot(gs[4, :])
    weights = results["weights"]
    avg_weights = weights.mean().sort_values(ascending=False).head(5)
    for asset in avg_weights.index:
        ax6.plot(weights.index, weights[asset], label=asset, linewidth=2)
    ax6.set_title("Top 5 Asset Weights Over Time", fontsize=14, fontweight="bold")
    ax6.set_xlabel("Date", fontsize=11)
    ax6.set_ylabel("Weight", fontsize=11)
    ax6.legend(loc="upper left", fontsize=10, ncol=5)
    ax6.grid(True, alpha=0.3)
    plt.suptitle(
        "MST Entropy Regime-Adaptive Strategy - Comprehensive Analysis",
        fontsize=18,
        fontweight="bold",
        y=0.995,
    )
    return fig


def plot_regime_performance(results, benchmark_returns):
    """
    Plot performance breakdown by regime

    Parameters:
    -----------
    results : dict
        Backtest results
    benchmark_returns : pd.Series
        Benchmark returns
    """
    portfolio_returns = results["portfolio_values_series"].pct_change().dropna()
    regimes = results["regimes"]
    regime_map = {}
    for date in portfolio_returns.index:
        closest_regime_date = (
            regimes.index[regimes.index <= date][-1]
            if any(regimes.index <= date)
            else regimes.index[0]
        )
        regime_map[date] = regimes.loc[closest_regime_date]
    returns_by_regime = pd.DataFrame({"Return": portfolio_returns, "Regime": pd.Series(regime_map)})
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    fig.suptitle("Performance by Market Regime", fontsize=18, fontweight="bold")
    ax1 = axes[0, 0]
    for regime in ["BULLISH", "NEUTRAL", "BEARISH"]:
        regime_returns = returns_by_regime[returns_by_regime["Regime"] == regime]["Return"]
        if len(regime_returns) > 0:
            ax1.hist(regime_returns * 100, bins=50, alpha=0.5, label=regime)
    ax1.set_title("Return Distribution by Regime", fontsize=14, fontweight="bold")
    ax1.set_xlabel("Daily Return (%)", fontsize=11)
    ax1.set_ylabel("Frequency", fontsize=11)
    ax1.legend(fontsize=11)
    ax1.grid(True, alpha=0.3)
    ax2 = axes[0, 1]
    for regime in ["BULLISH", "NEUTRAL", "BEARISH"]:
        regime_returns = returns_by_regime[returns_by_regime["Regime"] == regime]["Return"]
        if len(regime_returns) > 0:
            cum_returns = (1 + regime_returns).cumprod()
            ax2.plot(cum_returns.index, cum_returns.values, label=regime, linewidth=2)
    ax2.set_title("Cumulative Returns by Regime", fontsize=14, fontweight="bold")
    ax2.set_xlabel("Date", fontsize=11)
    ax2.set_ylabel("Cumulative Return", fontsize=11)
    ax2.legend(fontsize=11)
    ax2.grid(True, alpha=0.3)
    ax3 = axes[1, 0]
    sharpe_by_regime = {}
    for regime in ["BULLISH", "NEUTRAL", "BEARISH"]:
        regime_returns = returns_by_regime[returns_by_regime["Regime"] == regime]["Return"]
        if len(regime_returns) > 10:
            sharpe = regime_returns.mean() / regime_returns.std() * np.sqrt(252)
            sharpe_by_regime[regime] = sharpe
    colors = {"BULLISH": "green", "NEUTRAL": "gray", "BEARISH": "red"}
    bars = ax3.bar(
        sharpe_by_regime.keys(),
        sharpe_by_regime.values(),
        color=[colors[r] for r in sharpe_by_regime.keys()],
        alpha=0.7,
    )
    ax3.set_title("Sharpe Ratio by Regime", fontsize=14, fontweight="bold")
    ax3.set_ylabel("Sharpe Ratio", fontsize=11)
    ax3.grid(True, alpha=0.3, axis="y")
    ax3.axhline(y=0, color="black", linestyle="--", linewidth=1)
    ax4 = axes[1, 1]
    regime_changes = regimes != regimes.shift()
    regime_blocks = regime_changes.cumsum()
    regime_durations = {}
    for regime in ["BULLISH", "NEUTRAL", "BEARISH"]:
        regime_mask = regimes == regime
        durations = []
        for block_id in regime_blocks[regime_mask].unique():
            duration = (regime_blocks == block_id).sum()
            durations.append(duration)
        if durations:
            regime_durations[regime] = np.mean(durations)
    bars = ax4.bar(
        regime_durations.keys(),
        regime_durations.values(),
        color=[colors[r] for r in regime_durations.keys()],
        alpha=0.7,
    )
    ax4.set_title("Average Regime Duration", fontsize=14, fontweight="bold")
    ax4.set_ylabel("Days", fontsize=11)
    ax4.grid(True, alpha=0.3, axis="y")
    plt.tight_layout()
    return fig


if __name__ == "__main__":
    try:
        prices = download_all_data(
            CONFIG["ETF_UNIVERSE"] + [CONFIG["BENCHMARK"]], CONFIG["START_DATE"], CONFIG["END_DATE"]
        )
        if len(prices) < CONFIG["MIN_DATA_POINTS"]:
            raise ValueError(
                f"Insufficient data: {len(prices)} days < {CONFIG['MIN_DATA_POINTS']} required"
            )
    except Exception as e:
        print(f"FATAL ERROR in data download: {str(e)}")
        raise
    returns = prices.pct_change().dropna()
    print(f"Returns computed: {returns.shape}")
    print()
    benchmark_prices = prices[[CONFIG["BENCHMARK"]]]
    benchmark_returns = returns[[CONFIG["BENCHMARK"]]]
    strategy_assets = [c for c in prices.columns if c != CONFIG["BENCHMARK"]]
    strategy_prices = prices[strategy_assets]
    strategy_returns = returns[strategy_assets]
    print("=" * 80)
    print("DETECTING REGIMES USING MST ENTROPY")
    print("=" * 80)
    print()
    regimes, mst_metrics = detect_regime_mst(strategy_returns, CONFIG["REGIME_WINDOW"])
    regimes_smoothed = smooth_regimes(regimes, CONFIG["REGIME_MIN_DURATION"])
    print("Regime Detection Summary:")
    print("-" * 60)
    print(regimes_smoothed.value_counts())
    print()
    print("Regime transitions:")
    regime_changes = (regimes_smoothed != regimes_smoothed.shift()).sum()
    print(f"  Total transitions: {regime_changes}")
    print(f"  Average regime duration: {len(regimes_smoothed) / regime_changes:.1f} days")
    print()
    diagnose_regime_distribution(mst_metrics, regimes_smoothed)
    results = backtest_strategy(strategy_prices, strategy_returns, regimes_smoothed, CONFIG)
    print("=" * 80)
    print("PERFORMANCE ANALYSIS")
    print("=" * 80)
    print()
    strategy_portfolio_returns = results["portfolio_values_series"].pct_change().dropna()
    aligned_benchmark = benchmark_returns.reindex(strategy_portfolio_returns.index).dropna()
    strategy_aligned = strategy_portfolio_returns.reindex(aligned_benchmark.index)
    comparison = compare_with_benchmark(strategy_aligned, aligned_benchmark[CONFIG["BENCHMARK"]])
    print("STRATEGY METRICS:")
    print("-" * 60)
    for metric, value in comparison["Strategy"].items():
        if isinstance(value, float):
            if "Return" in metric or "Drawdown" in metric:
                print(f"  {metric:.<40} {value * 100:>8.2f}%")
            else:
                print(f"  {metric:.<40} {value:>8.4f}")
    print()
    print("BENCHMARK METRICS:")
    print("-" * 60)
    for metric, value in comparison["Benchmark"].items():
        if isinstance(value, float):
            if "Return" in metric or "Drawdown" in metric:
                print(f"  {metric:.<40} {value * 100:>8.2f}%")
            else:
                print(f"  {metric:.<40} {value:>8.4f}")
    print()
    print("RELATIVE PERFORMANCE:")
    print("-" * 60)
    print(f"  {'Alpha (annualized)':.<40} {comparison['Alpha'] * 100:>8.2f}%")
    print(f"  {'Beta':.<40} {comparison['Beta']:>8.4f}")
    print(f"  {'Information Ratio':.<40} {comparison['Information Ratio']:>8.4f}")
    print(f"  {'Tracking Error':.<40} {comparison['Tracking Error'] * 100:>8.2f}%")
    print()
    print("=" * 80)
    print("GENERATING VISUALIZATIONS")
    print("=" * 80)
    print()
    fig1 = plot_results(results, aligned_benchmark[CONFIG["BENCHMARK"]], mst_metrics)
    plt.show()
    fig2 = plot_regime_performance(results, aligned_benchmark[CONFIG["BENCHMARK"]])
    plt.show()
    print()
    print("=" * 80)
    print("BACKTEST COMPLETE")
    print("=" * 80)
